## Run test: Qwen/Qwen3.5-0.8B

Model page: https://huggingface.co/Qwen/Qwen3.5-0.8B

Task: `image-text-to-text`  ·  Library: `transformers`  ·  Source: official-template

> Reproduced from HuggingFace's official auto-generated colab/transformers snippet (identical code). Local inference on 8×W7900 with `device_map="auto"` added.

In [ ]:
%pip install -U transformers accelerate

In [ ]:
# HF Colab aligned image-text-to-text pipeline smoke test on one visible GPU.
import gc
import torch
from PIL import Image, ImageDraw
from transformers import pipeline

model_path = "Qwen/Qwen3.5-0.8B"
img = img  # Use in-memory image
img = Image.new("RGB", (224, 224), "white")
draw = ImageDraw.Draw(img)
draw.rectangle([50, 50, 170, 170], fill="black")
img.save(img_path)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": img_path},
            {"type": "text", "text": "What color is the square in the image?"},
        ],
    },
]
pipe = pipeline("image-text-to-text", model=model_path, device=0, trust_remote_code=True)
pipe_result = pipe(messages, max_new_tokens=40)
print(pipe_result)
pipe_tensor_devices = sorted({str(p.device) for p in pipe.model.parameters()})
print("pipe parameter devices =", pipe_tensor_devices)

del pipe
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# HF Colab aligned direct image inference smoke test.
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

processor = AutoProcessor.from_pretrained(model_path)
model = AutoModelForMultimodalLM.from_pretrained(model_path).to("cuda").eval()

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": img_path},
            {"type": "text", "text": "What color is the square in the image?"},
        ],
    },
]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))
